# ⚡ PowerCast: Household Electricity Forecasting
### Part 1: Data Cleaning & Preprocessing

**Author:** Rana Aljumayi
**Team:** Rana · Salwa · Renad · Raneem
**Le Wagon Data Science #2344 · Project Weeks**

---

**Goal of this notebook:**
Turn the raw minute-level data into a clean **hourly** dataset, split by date, ready for EDA and modeling.

**Contents:**
1. Mount Drive
2. Load Data
3. Convert Date & Time
4. Check Missing Values
5. Resample to Hourly
6. Fill Missing Hours
7. Sanity Check
8. Train / Validation / Test Split
9. Save Clean Data
10. Notes for the Team

**Output:** `data/hourly_clean.csv` · `data/split_dates.json`

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/PowerCast'
RAW = f'{BASE}/data/household_power_consumption.txt'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load Data

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(RAW, sep=';', na_values='?', low_memory=False)

print(df.shape)
df.head()

(2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 3. Convert Date & Time

In [3]:
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df = df.drop(columns=['Date', 'Time'])
df = df.set_index('datetime').sort_index()

print('Start:', df.index.min())
print('End:', df.index.max())
print('Duplicate timestamps:', df.index.duplicated().sum())
df.head()

Start: 2006-12-16 17:24:00
End: 2010-11-26 21:02:00
Duplicate timestamps: 0


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## 4. Check Missing Values

In [4]:
# missing values per column
print(df.isna().sum())
print('Missing %:', round(df['Global_active_power'].isna().mean() * 100, 2))

# missing timestamps (minutes not in the data)
full_range = pd.date_range(df.index.min(), df.index.max(), freq='min')
print('Expected rows:', len(full_range), '| Actual rows:', len(df))

# longest gap of missing values
na = df['Global_active_power'].isna()
gaps = na.groupby((~na).cumsum()).sum()
print('Longest missing gap (minutes):', gaps.max())
print('Longest missing gap (days):', round(gaps.max() / 1440, 1))

Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64
Missing %: 1.25
Expected rows: 2075259 | Actual rows: 2075259
Longest missing gap (minutes): 7226
Longest missing gap (days): 5.0


## 5. Resample to Hourly

In [5]:
df_hourly = df.resample('h').mean()

print(df_hourly.shape)
print('Missing hours:', df_hourly['Global_active_power'].isna().sum())

na_h = df_hourly['Global_active_power'].isna()
gaps_h = na_h.groupby((~na_h).cumsum()).sum()
print('Gaps by length (hours):')
print(gaps_h[gaps_h > 0].value_counts().sort_index())

(34589, 7)
Missing hours: 421
Gaps by length (hours):
Global_active_power
1      1
14     1
33     1
52     1
54     1
61     1
87     1
119    1
Name: count, dtype: int64


## 6. Fill Missing Hours

In [6]:
cols = df_hourly.columns
na_h = df_hourly['Global_active_power'].isna()
gap_len = na_h.groupby((~na_h).cumsum()).transform('sum')
short_gap = na_h & (gap_len <= 3)

df_clean = df_hourly.copy()

# short gaps (<= 3 hours): time interpolation
interp = df_hourly.interpolate(method='time')
df_clean.loc[short_gap, cols] = interp.loc[short_gap, cols]

# long gaps: same hour from the previous week
for _ in range(5):
    df_clean[cols] = df_clean[cols].fillna(df_clean[cols].shift(168))

df_clean['was_missing'] = na_h.astype(int)

print('Missing after filling:', df_clean[cols].isna().sum().sum())
print('Filled hours:', df_clean['was_missing'].sum())

Missing after filling: 0
Filled hours: 421


## 7. Sanity Check

In [7]:
print('Negative values:', (df_clean[cols] < 0).sum().sum())
df_clean[cols].describe().round(2)

Negative values: 0


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
count,34589.00,34589.00,34589.00,34589.00,34589.00,34589.00,34589.00
mean,1.09,0.12,240.85,4.62,1.12,1.30,6.45
std,0.90,0.07,3.00,3.75,3.54,4.19,7.35
min,0.12,0.00,225.83,0.50,0.00,0.00,0.00
25%,0.34,0.08,239.20,1.51,0.00,0.00,0.65
50%,0.80,0.11,240.99,3.41,0.00,0.33,1.65
75%,1.58,0.15,242.69,6.59,0.00,0.65,14.17
max,6.56,0.77,251.90,28.38,48.37,46.43,21.55


## 8. Train / Validation / Test Split

In [8]:
TRAIN_END = '2009-12-31 23:00'
VAL_END = '2010-06-30 23:00'

train = df_clean.loc[:TRAIN_END]
val = df_clean.loc['2010-01-01':VAL_END]
test = df_clean.loc['2010-07-01':]

for name, part in [('Train', train), ('Validation', val), ('Test', test)]:
    print(f'{name}: {part.index.min()} -> {part.index.max()} | {len(part)} hours')

Train: 2006-12-16 17:00:00 -> 2009-12-31 23:00:00 | 26671 hours
Validation: 2010-01-01 00:00:00 -> 2010-06-30 23:00:00 | 4344 hours
Test: 2010-07-01 00:00:00 -> 2010-11-26 21:00:00 | 3574 hours


## 9. Save Clean Data

In [9]:
import json

df_save = df_clean.copy()
df_save['split'] = 'train'
df_save.loc['2010-01-01':VAL_END, 'split'] = 'val'
df_save.loc['2010-07-01':, 'split'] = 'test'

df_save.to_csv(f'{BASE}/data/hourly_clean.csv')

splits = {'train_end': TRAIN_END, 'val_start': '2010-01-01 00:00',
          'val_end': VAL_END, 'test_start': '2010-07-01 00:00'}
with open(f'{BASE}/data/split_dates.json', 'w') as f:
    json.dump(splits, f, indent=2)

print(df_save['split'].value_counts())

split
train    26671
val       4344
test      3574
Name: count, dtype: int64


## 10. Notes for the Team

**Data:** 2,075,259 minute readings (2006-12-16 → 2010-11-26), resampled to **34,589 hourly rows**.

**Target:** `Global_active_power` (kW, hourly mean)

**Missing values:**
- 1.25% of minutes were missing (same rows in all columns). No missing timestamps.
- After resampling: 421 missing hours (1 short gap of 1h + 7 long gaps of 14–119h, longest ≈ 5 days).
- Short gaps (≤ 3h) → time interpolation.
- Long gaps → same hour from the previous week (uses past data only, no leakage).
- Column `was_missing` = 1 for the 421 filled hours.

**Sanity check:** no negative values. Power 0.12–6.56 kW (mean 1.09). Voltage 226–252 V. No values removed.

**Split (by date, no shuffle):**

| Split | From | To | Hours |
|---|---|---|---|
| Train | 2006-12-16 | 2009-12-31 | 26,671 |
| Validation | 2010-01-01 | 2010-06-30 | 4,344 |
| Test | 2010-07-01 | 2010-11-26 | 3,574 |

⚠️ Test is untouched until the final evaluation.

**Files (in `data/`):**
- `hourly_clean.csv` → clean hourly data + `split` column
- `split_dates.json` → split boundaries

**How to load:**
`df = pd.read_csv(f'{BASE}/data/hourly_clean.csv', index_col=0, parse_dates=True)`